# 04 — Per-product counterfactual validation

Does acting on the model beat the status quo? We estimate **each product's own**
elasticity, set its margin-maximizing price, and aggregate the counterfactual margin
uplift vs the prices actually charged.

Two honest problems and their fixes:
1. **Per-product slopes are noisy** (~13 months each, SD≈10) → partial-pool (empirical-Bayes
   shrinkage) toward the pooled elasticity.
2. **Constant-elasticity extrapolates** → cap extreme ε and constrain price moves to a ±25%
   guardrail (what real pricing teams do; keeps us near the observed price support).

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from src.data import load_prepared
from src.elasticity import per_product_elasticities
from src.optimize import per_product_counterfactual

df = load_prepared()
etab, eps_pool = per_product_elasticities(df)
print(f'pooled ε = {eps_pool:.2f} | products={len(etab)} estimable={etab.eps_raw.notna().sum()}')
etab.dropna(subset=['eps_raw']).round(2).head(8)

## Cost-sensitivity sweep
Unit cost isn't in the data, so we sweep it as a fraction of each product's reference price
and report the aggregate margin uplift (weighted by observed periods).

In [ ]:
sweep = pd.DataFrame([per_product_counterfactual(df, etab, cost_frac=cf)[1]
                      for cf in [0.4, 0.5, 0.6, 0.7]])
sweep.round(2)

In [ ]:
# per-product recommendations at a central cost assumption
pp, summary = per_product_counterfactual(df, etab, cost_frac=0.6)
print(summary)
print('\nrecommended price move:',
      'cut', int((pp.price_change_pct < -0.01).sum()),
      '| hold', int((pp.price_change_pct.abs() <= 0.01).sum()),
      '| raise', int((pp.price_change_pct > 0.01).sum()))
pp.sort_values('uplift_pct', ascending=False).round(2).head(10)

## Verdict
At a central cost assumption the model recommends mostly **modest price cuts**, holds price
on products it can't reliably optimize, and delivers a **positive aggregate margin uplift**
that stays positive across the whole cost sweep.

**Keep honest:** in-sample anchor; constant per-product elasticity; assumed proportional cost;
the ±25% guardrail does real work (without it, noisy elasticities over-extrapolate and the
uplift explodes). The number to trust is the *direction and rough magnitude*, not three sig figs.